## Importing required modules and the data

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_excel(r"D:\Learnerea\video content creation\creditrisk_with_pandas\raw data\roll rate data\synthetic_credit_roll_rate_12m.xlsx")
df.sample(5)

,customer_id,customer_name,account_id,product,sanctioned_limit,outstanding_balance,bill_generation_date,due_date,amount_due,payment_date,payment_amount,snapshot_date
1766,CUST4167,Aarav Bhardwaj,ACC40167,Auto Loan,1138000,656955.73,2026-06-07,2026-06-25,18547.78,NaT,0.00,2026-06-30
1490,CUST4091,Pooja Sethi,ACC40091,Credit Card,207000,46775.41,2026-05-08,2026-05-26,2304.38,NaT,0.00,2026-05-31
1319,CUST4120,Aditya Nair,ACC40120,Personal Loan,493000,310289.19,2026-04-08,2026-04-24,14277.62,NaT,0.00,2026-04-30
2118,CUST4119,Deepak Bhardwaj,ACC40119,Credit Card,390000,203503.85,2026-08-04,2026-08-23,9566.76,2026-08-06,9566.76,2026-08-31
1398,CUST4199,Reyansh Chopra,ACC40199,Credit Card,351000,100040.68,2026-04-03,2026-04-20,4782.70,2026-04-10,4782.70,2026-04-30


## Creating additional variables like - final_payment_date, dpd, buckets
#### These were already explained in the previous videos, you can refer to them for detailed explanation

In [3]:
# Creating final_payment_date and dpd variables here
df['final_payment_date'] = df['payment_date'].combine_first(df['snapshot_date'])
df['dpd'] = ((df['final_payment_date'] - df['due_date']).dt.days).clip(lower=0)

# Creating delinquency bucket here
conditions = [
                df['dpd'] <= 0,
                df['dpd'].between(1, 30),
                df['dpd'].between(31, 60),
                df['dpd'].between(61, 90)
            ]

category = ['bucket0','bucket1','bucket2','bucket3']

df['buckets'] = np.select(conditions, category, default='bucket4')
df.groupby('buckets')[['customer_id']].count()

,customer_id
buckets,
bucket0,2017
bucket1,156
bucket2,77
bucket3,54
bucket4,96


In [4]:
df.head()

,customer_id,customer_name,account_id,product,sanctioned_limit,outstanding_balance,bill_generation_date,due_date,amount_due,payment_date,payment_amount,snapshot_date,final_payment_date,dpd,buckets
0,CUST4001,Anjali Gupta,ACC40001,Auto Loan,899000,438629.22,2025-10-01,2025-10-19,11890.68,2025-10-05,11890.68,2025-10-31,2025-10-05,0,bucket0
1,CUST4002,Meera Mishra,ACC40002,Credit Card,384000,97837.25,2025-10-04,2025-10-19,4509.06,2025-10-09,4509.06,2025-10-31,2025-10-09,0,bucket0
2,CUST4003,Vivaan Bhardwaj,ACC40003,Home Loan,7211000,4272008.00,2025-10-07,2025-10-26,30104.19,2025-10-12,30104.19,2025-10-31,2025-10-12,0,bucket0
3,CUST4004,Diya Chopra,ACC40004,Personal Loan,226000,111831.38,2025-10-02,2025-10-21,4862.94,2025-10-09,4862.94,2025-10-31,2025-10-09,0,bucket0
4,CUST4005,Dhruv Pillai,ACC40005,Credit Card,327000,117954.55,2025-10-01,2025-10-18,6353.30,2025-10-04,6353.30,2025-10-31,2025-10-04,0,bucket0


# Further processing

#### Subset the data with required variables only - account_id, outstanding_balance, snapshot_date, buckets and sort the same

In [73]:
subset = df[['account_id', 'outstanding_balance', 'snapshot_date', 'buckets']].sort_values(['account_id','snapshot_date'])
subset.head()

,account_id,outstanding_balance,snapshot_date,buckets
0,ACC40001,438629.22,2025-10-31,bucket0
200,ACC40001,434862.13,2025-11-30,bucket0
400,ACC40001,431095.04,2025-12-31,bucket0
600,ACC40001,427327.95,2026-01-31,bucket0
800,ACC40001,423560.86,2026-02-28,bucket0


#### Creating next bucket, next month and next balance variables

In [77]:
subset['next_bucket'] = subset.groupby('account_id')['buckets'].shift(-1)
subset['next_month'] = subset.groupby('account_id')['snapshot_date'].shift(-1)
subset['next_bal'] = subset.groupby('account_id')['outstanding_balance'].shift(-1)
subset.head()

,account_id,outstanding_balance,snapshot_date,buckets,next_bucket,next_month,next_bal
0,ACC40001,438629.22,2025-10-31,bucket0,bucket0,2025-11-30,434862.13
200,ACC40001,434862.13,2025-11-30,bucket0,bucket0,2025-12-31,431095.04
400,ACC40001,431095.04,2025-12-31,bucket0,bucket0,2026-01-31,427327.95
600,ACC40001,427327.95,2026-01-31,bucket0,bucket0,2026-02-28,423560.86
800,ACC40001,423560.86,2026-02-28,bucket0,bucket0,2026-03-31,419793.78


#### Filter for only Oct & Nov month data

In [78]:
filtered_df = (subset[subset['snapshot_date']=='2025-10-31']).copy()
filtered_df.head()

,account_id,outstanding_balance,snapshot_date,buckets,next_bucket,next_month,next_bal
0,ACC40001,438629.22,2025-10-31,bucket0,bucket0,2025-11-30,434862.13
1,ACC40002,97837.25,2025-10-31,bucket0,bucket0,2025-11-30,116245.05
2,ACC40003,4272008.00,2025-10-31,bucket0,bucket0,2025-11-30,4231330.99
3,ACC40004,111831.38,2025-10-31,bucket0,bucket0,2025-11-30,110879.81
4,ACC40005,117954.55,2025-10-31,bucket0,bucket0,2025-11-30,138450.49


#### Get the transition count (cross tab) and percentage of no. accounts transitioned from one bucket to another during Oct to Nov

In [82]:
(pd.crosstab(filtered_df['buckets'],filtered_df['next_bucket'],normalize='index')*100).round(2)

next_bucket,bucket0,bucket1,bucket2,bucket3,bucket4
buckets,,,,,
bucket0,90.71,9.29,0.00,0.00,0.0
bucket1,50.00,16.67,33.33,0.00,0.0
bucket2,28.57,0.00,28.57,42.86,0.0
bucket3,0.00,0.00,0.00,0.00,100.0
bucket4,50.00,0.00,0.00,50.00,0.0


In [58]:
# transition_count_pct in percentage

In [83]:
# Crosstab on outstanding balance
(pd.crosstab(filtered_df['buckets'],filtered_df['next_bucket'],values=filtered_df['outstanding_balance'],aggfunc=sum,normalize='index')*100).round(2)

next_bucket,bucket0,bucket1,bucket2,bucket3,bucket4
buckets,,,,,
bucket0,96.31,3.69,0.00,0.00,0.0
bucket1,34.88,6.43,58.69,0.00,0.0
bucket2,71.32,0.00,8.79,19.88,0.0
bucket3,0.00,0.00,0.00,0.00,100.0
bucket4,1.08,0.00,0.00,98.92,0.0


# 11 Months analysis

In [84]:
subset_sorted = subset

In [85]:
subset_sorted['expected_next_month'] = (subset_sorted['snapshot_date'] + pd.offsets.MonthEnd(1))

migration_df = subset_sorted[subset_sorted['next_month'] == subset_sorted['expected_next_month']].copy()

migration_df.head()

,account_id,outstanding_balance,snapshot_date,buckets,next_bucket,next_month,next_bal,expected_next_month
0,ACC40001,438629.22,2025-10-31,bucket0,bucket0,2025-11-30,434862.13,2025-11-30
200,ACC40001,434862.13,2025-11-30,bucket0,bucket0,2025-12-31,431095.04,2025-12-31
400,ACC40001,431095.04,2025-12-31,bucket0,bucket0,2026-01-31,427327.95,2026-01-31
600,ACC40001,427327.95,2026-01-31,bucket0,bucket0,2026-02-28,423560.86,2026-02-28
800,ACC40001,423560.86,2026-02-28,bucket0,bucket0,2026-03-31,419793.78,2026-03-31


In [86]:

# Define the order of delinquency buckets
bucket_order = [
    'bucket0', 'bucket1', 'bucket2', 'bucket3', 'bucket4'
]

# Dictionary to store all monthly matrices
monthly_matrices = {}

# Loop through each starting snapshot month
for month, data in migration_df.groupby('snapshot_date', sort=True):

    # 1. Account transition count
    transition_count = pd.crosstab(
        data['buckets'],
        data['next_bucket']
    ).reindex(
        index=bucket_order,
        columns=bucket_order,
        fill_value=0
    )

    # 2. Account transition percentage
    transition_pct = (
        transition_count.div(
            transition_count.sum(axis=1).replace(0, np.nan),
            axis=0
        ) * 100
    ).round(2)

    # 3. Opening balance transition matrix
    transition_bal = pd.crosstab(
        data['buckets'],
        data['next_bucket'],
        values=data['outstanding_balance'],
        aggfunc='sum'
    ).reindex(
        index=bucket_order,
        columns=bucket_order,
        fill_value=0
    )

    # 4. Opening balance transition percentage
    transition_bal_pct = (
        transition_bal.div(
            transition_bal.sum(axis=1).replace(0, np.nan),
            axis=0
        ) * 100
    ).round(2)

    # 5. Closing balance transition matrix (optional)
    transition_next_bal = pd.crosstab(
        data['buckets'],
        data['next_bucket'],
        values=data['next_bal'],
        aggfunc='sum'
    ).reindex(
        index=bucket_order,
        columns=bucket_order,
        fill_value=0
    )

    # Store all matrices for this transition
    next_month = month + pd.offsets.MonthEnd(1)

    transition_name = (
        f"{month:%b_%y}_to_{next_month:%b_%y}"
    )

    monthly_matrices[transition_name] = {
        'count': transition_count,
        'count_pct': transition_pct,
        'opening_balance': transition_bal,
        'opening_balance_pct': transition_bal_pct,
        'closing_balance': transition_next_bal
    }

In [87]:

roll_forward_summary = []

for month, matrices in monthly_matrices.items():

    pct = matrices['count_pct']

    roll_forward_summary.append({
        'Transition': month,

        'Bucket 0': pct.loc['bucket0',
                            ['bucket1', 'bucket2', 'bucket3', 'bucket4']].sum(),

        'Bucket 1': pct.loc['bucket1',
                            ['bucket2', 'bucket3', 'bucket4']].sum(),

        'Bucket 2': pct.loc['bucket2',
                            ['bucket3', 'bucket4']].sum(),

        'Bucket 3': pct.loc['bucket3',
                            ['bucket4']].sum(),

        'Bucket 4': pct.loc['bucket4', 'bucket4']
    })

roll_forward_df = pd.DataFrame(roll_forward_summary)

roll_forward_df = roll_forward_df.round(2)

roll_forward_df


,Transition,Bucket 0,Bucket 1,Bucket 2,Bucket 3,Bucket 4
0,Oct_25_to_Nov_25,9.29,33.33,42.86,100.00,0.00
1,Nov_25_to_Dec_25,6.98,38.89,50.00,25.00,50.00
2,Dec_25_to_Jan_26,4.65,46.67,75.00,66.67,50.00
3,Jan_26_to_Feb_26,6.94,22.22,57.14,37.50,100.00
4,Feb_26_to_Mar_26,6.32,33.33,25.00,75.00,83.33
5,Mar_26_to_Apr_26,6.94,66.67,40.00,50.00,75.00
6,Apr_26_to_May_26,5.33,58.33,66.67,100.00,85.71
7,May_26_to_Jun_26,10.30,16.67,71.43,42.86,88.89
8,Jun_26_to_Jul_26,6.41,47.62,100.00,55.56,100.00
9,Jul_26_to_Aug_26,8.18,27.27,45.45,33.33,81.25
